# Investigation 01 — v1 vs v2 paired comparison

This notebook tests the effect of the v2 abstention prompt as a paired intervention.

Instead of comparing aggregate averages only, it aligns v1 and v2 predictions for the same:

`provider × model × prompt family × condition × sentence × repetition`

This makes it possible to answer:

- Which exact v1 errors were fixed by v2?
- Did v2 introduce any new errors?
- Did v2 mainly improve controls, or also improve schema-present examples?
- Was the gain stronger for direct-schema prompting or structured-role prompting?

Thesis use: this directly supports the claim that the v2 schema-present/NONE mechanism is an experimental intervention rather than a minor prompt wording change.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Run from the project notebooks/ directory.
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data"
OUTPUTS_DIR = DATA_DIR / "outputs"
PARSED_PATH = OUTPUTS_DIR / "parsed_responses.jsonl"
RAW_PATH = OUTPUTS_DIR / "raw_responses.jsonl"
GOLD_PATH = DATA_DIR / "gold" / "sentences_v1.jsonl"

EXPORT_DIR = OUTPUTS_DIR / "top4_investigations"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def read_jsonl(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f"Invalid JSON in {path} line {line_no}: {exc}") from exc
    return pd.DataFrame(rows)

def safe_read_jsonl(path: Path) -> pd.DataFrame:
    return read_jsonl(path) if path.exists() else pd.DataFrame()

def prompt_generation(prompt_id) -> str:
    prompt_id = str(prompt_id)
    if "v2" in prompt_id or "abstention" in prompt_id:
        return "v2_abstention"
    if "v1" in prompt_id:
        return "v1"
    return "unknown"

def prompt_base(prompt_id: str) -> str:
    prompt_id = str(prompt_id)
    if "direct_schema" in prompt_id:
        return "direct_schema"
    if "structured_roles" in prompt_id:
        return "structured_roles"
    if "naive" in prompt_id:
        return "naive"
    return "unknown"

def condition_family_from_id(condition_id) -> str:
    condition_id = str(condition_id)
    if "temp_0" in condition_id:
        return "temp_0"
    if "temp_03" in condition_id:
        return "temp_03"
    if "temp_07" in condition_id:
        return "temp_07"
    return condition_id

def add_derived_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["prompt_generation"] = out["prompt_id"].map(prompt_generation) if "prompt_id" in out.columns else "unknown"
    out["prompt_base"] = out["prompt_id"].map(prompt_base) if "prompt_id" in out.columns else "unknown"
    out["condition_family_short"] = out["condition_id"].map(condition_family_from_id) if "condition_id" in out.columns else "unknown"
    out["is_control"] = out["sentence_type"].eq("control_weak_schema") if "sentence_type" in out.columns else False
    out["is_non_control"] = ~out["is_control"]

    if "schema_present" not in out.columns:
        out["schema_present"] = np.where(out.get("main_image_schema", pd.Series()).eq("NONE"), "no", "yes")
    out["gold_schema_present"] = np.where(out["is_control"], "no", "yes")

    out["schema_present_correct"] = out["schema_present"].eq(out["gold_schema_present"])
    out["primary_schema_correct"] = out["main_image_schema"].eq(out["expected_schema_primary"])
    out["lm_correct"] = out["literal_or_metaphorical"].eq(out["expected_literal_or_metaphorical"])
    out["control_correct"] = out["is_control"] & out["literal_or_metaphorical"].eq("control") & out["main_image_schema"].eq("NONE")
    out["control_false_positive_schema"] = out["is_control"] & out["main_image_schema"].notna() & ~out["main_image_schema"].eq("NONE")
    out["predicted_none"] = out["main_image_schema"].eq("NONE") | out["literal_or_metaphorical"].eq("control") | out["schema_present"].eq("no")
    return out

def pct(x, digits=1):
    if x is None or pd.isna(x):
        return "NA"
    return f"{100*x:.{digits}f}%"

def save_csv(df: pd.DataFrame, filename: str) -> Path:
    path = EXPORT_DIR / filename
    df.to_csv(path, index=False)
    print(f"Wrote: {path}")
    return path

def display_percent_table(df: pd.DataFrame, percent_cols: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in percent_cols:
        if col in out.columns:
            out[col] = out[col].map(lambda x: pct(x) if x is not None else "NA")
    return out

parsed_all = add_derived_columns(read_jsonl(PARSED_PATH))
structured = parsed_all[parsed_all["parse_status"].eq("parsed")].copy()

print(f"All parsed records: {len(parsed_all)}")
print(f"Structured records: {len(structured)}")

In [ ]:
# Keep only v1/v2 prompts where a paired comparison is meaningful.
pairable = structured[structured["prompt_base"].isin(["direct_schema", "structured_roles"])].copy()

key_cols = [
    "provider", "model_id", "prompt_base", "condition_id", 
    "sentence_id", "sentence_type", "expected_schema_primary",
    "expected_literal_or_metaphorical", "repetition_index"
]

v1 = pairable[pairable["prompt_generation"].eq("v1")].copy()
v2 = pairable[pairable["prompt_generation"].eq("v2_abstention")].copy()

keep_cols = key_cols + [
    "prompt_id", "schema_present", "literal_or_metaphorical", "main_image_schema",
    "schema_present_correct", "primary_schema_correct", "lm_correct",
    "control_correct", "control_false_positive_schema"
]

paired = v1[keep_cols].merge(
    v2[keep_cols],
    on=key_cols,
    how="inner",
    suffixes=("_v1", "_v2"),
)

print(f"v1 records: {len(v1)}")
print(f"v2 records: {len(v2)}")
print(f"paired records: {len(paired)}")
display(paired.head())

In [ ]:
# Overall paired deltas.
paired_summary = []

for group_cols in [[], ["prompt_base"], ["provider"], ["provider", "prompt_base"], ["sentence_type"], ["prompt_base", "sentence_type"]]:
    if group_cols:
        grouped = paired.groupby(group_cols, dropna=False)
    else:
        grouped = [("ALL", paired)]
    for keys, g in grouped:
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: key for col, key in zip(group_cols, keys)}
        row["n"] = len(g)
        for metric in ["schema_present_correct", "primary_schema_correct", "lm_correct", "control_correct", "control_false_positive_schema"]:
            row[f"{metric}_v1"] = g[f"{metric}_v1"].mean()
            row[f"{metric}_v2"] = g[f"{metric}_v2"].mean()
            row[f"{metric}_delta_v2_minus_v1"] = row[f"{metric}_v2"] - row[f"{metric}_v1"]
        paired_summary.append(row)

paired_summary = pd.DataFrame(paired_summary)
save_csv(paired_summary, "paired_v1_v2_summary.csv")
display(display_percent_table(paired_summary, [c for c in paired_summary.columns if c.endswith("_v1") or c.endswith("_v2") or c.endswith("_delta_v2_minus_v1")]))

In [ ]:
# Fix/worsen transition tables for key metrics.
def transition_table(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    tab = pd.crosstab(df[f"{metric}_v1"], df[f"{metric}_v2"], rownames=[f"{metric}_v1"], colnames=[f"{metric}_v2"])
    return tab

for metric in ["control_correct", "control_false_positive_schema", "primary_schema_correct", "lm_correct", "schema_present_correct"]:
    print(f"\nTransition table: {metric}")
    display(transition_table(paired, metric))

In [ ]:
# Classify paired changes into interpretable categories.
paired_changes = paired.copy()

paired_changes["control_change_type"] = np.select(
    [
        paired_changes["is_control"] if "is_control" in paired_changes.columns else paired_changes["sentence_type"].eq("control_weak_schema"),
        paired_changes["sentence_type"].eq("control_weak_schema") & paired_changes["control_false_positive_schema_v1"] & ~paired_changes["control_false_positive_schema_v2"],
        paired_changes["sentence_type"].eq("control_weak_schema") & ~paired_changes["control_false_positive_schema_v1"] & paired_changes["control_false_positive_schema_v2"],
        paired_changes["sentence_type"].eq("control_weak_schema") & paired_changes["control_correct_v1"] & paired_changes["control_correct_v2"],
        paired_changes["sentence_type"].eq("control_weak_schema") & paired_changes["control_false_positive_schema_v1"] & paired_changes["control_false_positive_schema_v2"],
    ],
    [
        "control_other",
        "v2_fixed_control_false_positive",
        "v2_introduced_control_false_positive",
        "control_correct_both",
        "control_false_positive_both",
    ],
    default="non_control",
)

# The first condition in np.select above is too broad; recode with ordered explicit logic.
def classify_control_change(row):
    if row["sentence_type"] != "control_weak_schema":
        return "non_control"
    if row["control_false_positive_schema_v1"] and not row["control_false_positive_schema_v2"]:
        return "v2_fixed_control_false_positive"
    if not row["control_false_positive_schema_v1"] and row["control_false_positive_schema_v2"]:
        return "v2_introduced_control_false_positive"
    if row["control_correct_v1"] and row["control_correct_v2"]:
        return "control_correct_both"
    if row["control_false_positive_schema_v1"] and row["control_false_positive_schema_v2"]:
        return "control_false_positive_both"
    return "control_other"

paired_changes["control_change_type"] = paired_changes.apply(classify_control_change, axis=1)

change_counts = (
    paired_changes.groupby(["prompt_base", "provider", "control_change_type"])
    .size()
    .reset_index(name="count")
    .sort_values(["prompt_base", "provider", "control_change_type"])
)

save_csv(change_counts, "paired_v1_v2_control_change_counts.csv")
display(change_counts)

In [ ]:
# Export the most useful sentence-level lists for thesis examples.
fixed_controls = paired_changes[paired_changes["control_change_type"].eq("v2_fixed_control_false_positive")].copy()
introduced_controls = paired_changes[paired_changes["control_change_type"].eq("v2_introduced_control_false_positive")].copy()
still_wrong_controls = paired_changes[paired_changes["control_change_type"].eq("control_false_positive_both")].copy()

save_csv(fixed_controls, "paired_examples_v2_fixed_control_false_positives.csv")
save_csv(introduced_controls, "paired_examples_v2_introduced_control_false_positives.csv")
save_csv(still_wrong_controls, "paired_examples_control_false_positive_both.csv")

print("Examples fixed by v2:")
display(fixed_controls[[
    "provider", "model_id", "prompt_base", "sentence_id", "expected_schema_primary",
    "main_image_schema_v1", "main_image_schema_v2",
    "literal_or_metaphorical_v1", "literal_or_metaphorical_v2"
]].head(20))

In [ ]:
# Visualise v1-v2 control false-positive reductions.
plot_df = paired_summary[
    paired_summary.get("prompt_base", pd.Series(index=paired_summary.index)).notna()
    & paired_summary.get("provider", pd.Series(index=paired_summary.index)).notna()
    & paired_summary.get("sentence_type", pd.Series(index=paired_summary.index)).eq("control_weak_schema")
].copy()

if not plot_df.empty:
    plot_df["label"] = plot_df["provider"] + " / " + plot_df["prompt_base"]
    ax = plot_df.plot(kind="bar", x="label", y="control_false_positive_schema_delta_v2_minus_v1", legend=False)
    ax.axhline(0)
    ax.set_title("v2 change in control false-positive schema rate")
    ax.set_xlabel("Provider / prompt family")
    ax.set_ylabel("Delta v2 - v1")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

## Thesis interpretation prompts

Use this notebook to make a precise claim such as:

> The v2 abstention prompts reduced false-positive schema assignment on the exact same control items, demonstrating that the improvement is not caused by a different dataset split but by a changed prompt intervention.

Important follow-up:
- Quote 3–5 examples where v1 assigned a schema and v2 correctly returned `NONE`.
- Quote 2–3 examples where v2 still over-attributed a schema; these are theoretically valuable residual errors.